In [1]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"사용 중인 GPU: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1
CUDA 사용 가능 여부: True
사용 중인 GPU: NVIDIA RTX A4000


In [2]:
# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 텍스트 탐색용
import re
from collections import Counter

# 머신러닝 (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# 모델
from xgboost import XGBClassifier

# 희소 행렬(sparse matrix) 타입 확인용
from scipy import sparse

# 그래프에서 한글이 깨지지 않도록 폰트 설정 (Windows 기준: 맑은 고딕)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

#plt 그래프 스타일 설정
plt.style.use("ggplot")

# 노트북 안에서 그래프가 바로 보이도록 설정
%matplotlib inline

print("라이브러리 불러오기 완료")

라이브러리 불러오기 완료


In [3]:
# ---- 파일 경로 설정 ----
# 노트북과 같은 폴더에 있는 파일명을 기준으로 경로를 설정합니다.
# 파일을 다른 폴더에 저장했거나 파일명이 다르면, 아래 값을 실제 파일 위치에 맞게 수정합니다.
# 이후 코드들은 파일명을 직접 쓰지 않고 이 변수들을 사용하므로, 경로 변경은 이 셀만 수정하면 됩니다.
TRAIN_PATH = "security_log_train.csv"             # 학습 데이터 (로그 텍스트 + 정답 level)
TEST_PATH = "security_log_test.csv"               # 테스트 데이터 (정답 level 없음)
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"  # 제출 양식
OUTPUT_PATH = "submission.csv"                     # 생성할 제출 파일

# ---- 실행 옵션 ----
# RANDOM_STATE: 결과 재현성을 위한 난수 시드입니다.
# 같은 시드를 사용하면 train/valid 분리나 모델 학습 결과가 매번 동일하게 재현됩니다.
RANDOM_STATE = 42

print("설정 완료 (전체 train 데이터를 사용합니다)")

설정 완료 (전체 train 데이터를 사용합니다)


In [4]:
# 위에서 정의한 경로 변수를 사용해 세 개의 데이터를 불러옵니다.
# train: 모델이 학습할 로그 텍스트(full_log)와 정답 라벨(level)이 들어 있습니다.
# test : 예측 대상 데이터입니다. 정답 level이 없으므로 우리가 예측해서 채웁니다.
# sample_submission: 제출 파일의 형식(행 개수, 컬럼)을 알려주는 양식입니다.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

# 각 데이터의 행/열 개수를 확인합니다. (train과 test의 행 개수는 보통 다릅니다.)
print("train shape:", train.shape)
print("test shape :", test.shape)
print("sample_submission shape:", sample_submission.shape)

train shape: (472972, 3)
test shape : (1418916, 2)
sample_submission shape: (1418916, 2)


In [5]:
# 제출 양식(sample_submission) 확인 — 제출 파일은 이 형식과 똑같아야 합니다.
print("sample_submission columns:", list(sample_submission.columns))
display(sample_submission.head())

# 제출 파일의 행 개수는 test 데이터의 행 개수와 반드시 같아야 합니다.
assert len(sample_submission) == len(test), "sample_submission과 test의 행 개수가 다릅니다."
print("확인 완료: sample_submission과 test의 행 개수가 같습니다.")

sample_submission columns: ['id', 'level']


,id,level
0,1000000,0
1,1000001,0
2,1000002,0
3,1000003,0
4,1000004,0


확인 완료: sample_submission과 test의 행 개수가 같습니다.


In [6]:
def text_mining(df):
    df_ = df.copy()
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'(\\n)',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[^a-zA-Zㄱ-ㅣ가-힣0-9:=\s\(\)./,\<\>]+','',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r' ?(?P<note>[:=\(\)./,\<\>]) ?', ' \g<note> ', x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[0-9]+','<num>',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\s+',' ',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'[a-zA-Z]+\<','<eng><',x))
    df_['full_log'] = df_['full_log'].apply(lambda x: re.sub(r'\>[a-zA-Z]+','><eng>',x))

    return df_

In [7]:
train = text_mining(train)
test = text_mining(test)

In [ ]:
# test = pd.read_csv(TEST_PATH)

In [14]:
# =========================================================================
# [PRO_TIP] 최종 학습 전, 모델이 어떤 레벨을 헷갈려하는지 분석하는 검증 셀
# =========================================================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("🔍 [검증 단계] 로컬 데이터 분할 및 약점 분석을 시작합니다...")

# 1. 학습 데이터를 8:2로 쪼개서 가상 검증 데이터(y_val)를 만듭니다.
X_tr, X_val, y_tr, y_val = train_test_split(
    train["full_log"], 
    train["level"], 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=train["level"] # 클래스 비율을 동일하게 유지
)

# 2. 검증 전용 임시 TF-IDF 변환
val_tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3, max_df=0.95)
X_tr_tfidf = val_tfidf.fit_transform(X_tr)
X_val_tfidf = val_tfidf.transform(X_val)

# 3. 탐색 속도를 위해 cv=3으로 임시 모델 학습 (CPU 풀 가동)
val_base_model = LinearSVC(C=6.0, loss="squared_hinge", max_iter=10000, random_state=RANDOM_STATE, class_weight='balanced')
val_model = CalibratedClassifierCV(estimator=val_base_model, method="sigmoid", cv=3, n_jobs=-1)
val_model.fit(X_tr_tfidf, y_tr)

# 4. 검증 데이터 예측 및 성능 리포트 출력
val_proba = val_model.predict_proba(X_val_tfidf)
base_val_preds = np.argmax(val_proba, axis=1)

print("\n📊 [레벨별 Precision / Recall 확인 리포트]")
print(classification_report(y_val, base_val_preds, digits=4))
print("=" * 75)

🔍 [검증 단계] 로컬 데이터 분할 및 약점 분석을 시작합니다...

📊 [레벨별 Precision / Recall 확인 리포트]
              precision    recall  f1-score   support

           0     0.9979    0.9989    0.9984     66813
           1     0.9975    0.9954    0.9965     26504
           2     1.0000    1.0000    1.0000         2
           3     1.0000    0.9928    0.9964       828
           4     1.0000    1.0000    1.0000         2
           5     0.9594    0.9572    0.9583       444
           6     1.0000    0.5000    0.6667         2

    accuracy                         0.9977     94595
   macro avg     0.9935    0.9206    0.9452     94595
weighted avg     0.9977    0.9977    0.9977     94595



In [8]:
# 검증 단계에서는 데이터의 80%만 학습에 사용했습니다.
# 제출용 모델은 가지고 있는 학습 데이터를 모두 사용해 다시 학습하는 것이 일반적으로 더 좋습니다.

# 1) 최종 학습에 사용할 전체 입력/정답 (train_model 전체)
X_all = train["full_log"]
y_all = train["level"]

# 2) TF-IDF를 전체 학습 데이터로 다시 fit 합니다. (검증 때와 동일한 설정)
final_tfidf = TfidfVectorizer(
    max_features=10000,     # 사용할 단어(feature) 개수를 5000개로 제한 (baseline 기준)
    ngram_range=(1, 2),    # 단어 1개(unigram) + 2개 묶음(bigram)까지 사용
    min_df=3,              # 너무 드문 단어(3개 미만 문서에만 등장)는 제외
    max_df=0.95, 
)
# 학습 데이터는 fit_transform으로 단어 사전을 만들며 변환합니다.
X_all_tfidf = final_tfidf.fit_transform(X_all)
# 테스트 데이터는 transform만 사용합니다. fit_transform을 쓰면 데이터 누수가 되므로 주의합니다.
X_test_tfidf = final_tfidf.transform(test["full_log"])

print("X_all_tfidf  shape:", X_all_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_all_tfidf  shape: (472972, 10000)
X_test_tfidf shape: (1418916, 10000)


In [9]:
import numpy as np
import pandas as pd
import os
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# 0. 디렉토리 설정
os.makedirs("./submission", exist_ok=True)

# 1. 초고속 선형 SVM 모델 및 확률 보정기 정의 후 학습
base_model = LinearSVC(C=6.0, loss="squared_hinge", max_iter=10000, random_state=RANDOM_STATE, class_weight='balanced')
model = CalibratedClassifierCV(estimator=base_model, method="sigmoid", cv=3, n_jobs=-1)

model.fit(X_all_tfidf, y_all)
print("👉 [1단계] SVM 모델 학습 및 확률 보정 완료!")

# 2. 테스트 데이터의 클래스별 예측 확률 도출
test_final_proba = model.predict_proba(X_test_tfidf) # 실전에서는 X_test_tfidf 사용

base_preds = np.argmax(test_final_proba, axis=1) # 임계값 적용 전 기본 예측값 (0~6)
max_probas = np.max(test_final_proba, axis=1)    # SVM이 내린 가장 높은 확신의 확률 값

# =========================================================================
# 3. 🔥 [수정] 레벨별 커스텀 임계값(Dictionary) 정의
# =========================================================================
# 💡 이 수치들을 가감하며 리더보드 스코어를 튜닝해보세요!
custom_thresholds = {
    0: 0.55,  # 무난함. 대다수를 차지하므로 55% 미만의 애매한 건 7번으로 토스
    1: 0.60,  # 0번보다 살짝 불안정하므로 컷트라인을 60%로 상향
    2: 0.35,  # 🔥 초비상! 데이터가 2건뿐이라 컷트라인을 낮춰서 무조건 사수
    3: 0.90,  # 겉보기엔 완벽하지만, 실전(Test)에서 7번이 가장 많이 숨어드는 변장 구역
    4: 0.35,  # 🔥 초비상! 데이터가 2건뿐이므로 낮춰서 사수
    5: 0.75,  # ⚡ 현재 모델의 최대 약점(Precision 95%). 애매한 5번은 7번일 확률이 높음!
    6: 0.35   # 🔥 초비상! 이미 Recall이 0.50으로 반토막 난 상태라 컷트라인을 대폭 낮춰야 함
}

print("\n🔍 SVM 기반 레벨별 커스텀 임계값 후처리 적용 중...")
print("-" * 75)

👉 [1단계] SVM 모델 학습 및 확률 보정 완료!

🔍 SVM 기반 레벨별 커스텀 임계값 후처리 적용 중...
---------------------------------------------------------------------------


In [10]:
# =========================================================================
# 4. 🔥 [수정] 레벨별 맞춤형 컷트라인 적용 및 분포 확인
# =========================================================================
# 각 데이터의 예측값(base_preds)에 맞는 임계값을 매핑하여 배열로 만듭니다.
thresh_array = np.array([custom_thresholds[p] for p in base_preds])

# [핵심 원리] 1등 예측 확률(max_probas)이 해당 레벨의 컷트라인(thresh_array)보다 낮다면 -> '7'로 변경!
final_preds_custom = np.where(max_probas < thresh_array, 7, base_preds)

# 클래스별 분포 카운트
unique, counts = np.unique(final_preds_custom, return_counts=True)
counts_dict = dict(zip(unique, counts))

lvl7_count = counts_dict.get(7, 0)
lvl7_ratio = (lvl7_count / len(final_preds_custom)) * 100

print(f"▶️ level 7 생성 결과: {lvl7_count:<15,} 건 | {lvl7_ratio:.3f} %")
print("\n[전체 클래스 최종 분포 가이드]")
for k in sorted(counts_dict.keys()):
    print(f" - Level {k} : {counts_dict[k]:,} 건") # <--- :, 뒤의 공백 제거 완료!
print("-" * 75)

# =========================================================================
# 5. [수정] 커스텀 제출 파일 생성
# =========================================================================
th_sub_df = pd.DataFrame({
    "id": test["id"], 
    "level": final_preds_custom
})

file_name = "./submission/submission_svm_custom_th.csv"
th_sub_df.to_csv(file_name, index=False)

print(f"🎉 [실험 완료] 레벨별 후처리가 반영된 파일이 저장되었습니다! ({file_name})")

▶️ level 7 생성 결과: 1,343           건 | 0.095 %

[전체 클래스 최종 분포 가이드]
 - Level 0 : 1,002,089 건
 - Level 1 : 396,134 건
 - Level 2 : 34 건
 - Level 3 : 12,877 건
 - Level 4 : 34 건
 - Level 5 : 6,378 건
 - Level 6 : 27 건
 - Level 7 : 1,343 건
---------------------------------------------------------------------------
🎉 [실험 완료] 레벨별 후처리가 반영된 파일이 저장되었습니다! (./submission/submission_svm_custom_th.csv)
